# AI Lab: $\mathcal{R}_0$ Uncertainty Propagation from Generation-Time Distribution

**Book companion exercise**

**Considerations exercised:** 10 (generation-time misspecification)
**Estimated runtime:** ~25 minutes
**Companion audit:** [`17_generation_time_misspecification.md`](../ai-audit/17_generation_time_misspecification.md)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/exercises/ai-lab/17_R0_uncertainty_lab.ipynb)


## What this lab does

Quantifies how uncertainty in the generation-time distribution propagates through the Lotka-Euler $\mathcal{R}_0$ estimator. Compares three estimator shapes (exponential, gamma, point-mass) and Monte-Carlo-propagates uncertainty in the mean generation time. Confirms that early-pandemic $\mathcal{R}_0$ point estimates are misleading without uncertainty quantification.

## Setup

In [ ]:
import sys, os
# AI Lab notebooks live in exercises/ai-lab/ and need access to shared/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'python')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import math
from shared import book_style, BOOK_COLORS
from shared.parameters import baseline_chapter_07
book_style()
rng = np.random.default_rng(42)


## Step 1 — Three generation-time distributions with the same mean

Mean $\bar T_g = 5.2$ days, growth rate $r = 0.18$/day.

In [ ]:
from scipy.stats import expon, gamma as gamma_dist
from scipy.integrate import quad

T_g_mean = 5.2
r = 0.18

# Distribution choices
def g_exponential(tau):
    return expon.pdf(tau, scale=T_g_mean)

def g_gamma_shape4(tau):
    # Mean = shape * scale -> scale = mean / shape
    return gamma_dist.pdf(tau, a=4.0, scale=T_g_mean / 4.0)

def g_point_mass(tau):
    # Approximate point mass with a very narrow gamma
    return gamma_dist.pdf(tau, a=400.0, scale=T_g_mean / 400.0)

# Lotka-Euler conversion
def R0_from_g(g_func, r):
    integral, _ = quad(lambda tau: math.exp(-r*tau) * g_func(tau), 0, 50)
    return 1.0 / integral

# Simple-formula approximation: R_0 = 1 + r * T_g
R0_simple = 1 + r * T_g_mean
R0_exp = R0_from_g(g_exponential, r)
R0_gamma = R0_from_g(g_gamma_shape4, r)
R0_point = R0_from_g(g_point_mass, r)

print(f'Simple R_0 = 1 + r*T_g       = {R0_simple:.3f}')
print(f'Lotka-Euler with exponential = {R0_exp:.3f}  (matches simple formula)')
print(f'Lotka-Euler with gamma(k=4)  = {R0_gamma:.3f}  (peaked distribution)')
print(f'Lotka-Euler with point mass  = {R0_point:.3f}  (most peaked)')
print()
print(f'Spread: {R0_simple:.2f} -> {R0_point:.2f}  (factor {R0_point/R0_simple:.2f})')


## Step 2 — Monte-Carlo propagation of $T_g$ uncertainty

$\bar T_g \sim \text{Normal}(5.2, 1.0)$ truncated to positive values; use the gamma(shape=4) distribution shape:

In [ ]:
n_mc = 5000
T_g_samples = rng.normal(loc=5.2, scale=1.0, size=n_mc)
T_g_samples = np.maximum(T_g_samples, 0.5)

R0_samples = np.zeros(n_mc)
for i, T_g in enumerate(T_g_samples):
    g_local = lambda tau, T_g=T_g: gamma_dist.pdf(tau, a=4.0, scale=T_g/4.0)
    R0_samples[i] = R0_from_g(g_local, r)

print(f'R_0 distribution from MC propagation:')
print(f'  Mean   = {R0_samples.mean():.3f}')
print(f'  Median = {np.median(R0_samples):.3f}')
print(f'  95% CI = [{np.percentile(R0_samples, 2.5):.2f}, {np.percentile(R0_samples, 97.5):.2f}]')


## Step 3 — Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.hist(R0_samples, bins=40, color=BOOK_COLORS['primary'], alpha=0.85,
        edgecolor='white')
ax.axvline(R0_samples.mean(), color='red', lw=1.6, ls='--',
           label=f'mean = {R0_samples.mean():.2f}')
ax.axvline(np.percentile(R0_samples, 2.5), color='gray', lw=1.0, ls=':',
           label='95% CI')
ax.axvline(np.percentile(R0_samples, 97.5), color='gray', lw=1.0, ls=':')
ax.axvline(1.0, color='black', lw=0.8)
ax.set_xlabel(r'$\mathcal{R}_0$ estimate')
ax.set_ylabel('MC sample frequency')
ax.set_title('Monte-Carlo uncertainty in $\\mathcal{R}_0$ from generation-time uncertainty')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verification
assert R0_samples.mean() > 1.5
assert np.percentile(R0_samples, 97.5) - np.percentile(R0_samples, 2.5) > 0.3
print('\nVerified: meaningful uncertainty range emerges from realistic T_g uncertainty.')


## What's next

- Real early-pandemic estimates also need to account for uncertainty in $r$ itself; full Bayesian propagation is recommended (see Cori et al. 2013)
- Companion audit: `../ai-audit/17_generation_time_misspecification.md`
- Book Chapter 15 §15.4 develops the Lotka-Euler framework